# Feature Engineering

## Objective

In this notebook, we will create new business features from the existing data.

These features will help answer business questions related to:

- Sales Trends
- Customer Behaviour
- Delivery Performance
- Seller Performance
- Customer Satisfaction

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns",None)

In [2]:
customers = pd.read_csv("../Data/olist_customers_dataset.csv")
orders = pd.read_csv("../Data/olist_orders_dataset.csv")
order_items = pd.read_csv("../Data/olist_order_items_dataset.csv")
payments = pd.read_csv("../Data/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../Data/olist_order_reviews_dataset.csv")
products = pd.read_csv("../Data/olist_products_dataset.csv")
sellers = pd.read_csv("../Data/olist_sellers_dataset.csv")
geolocation = pd.read_csv("../Data/olist_geolocation_dataset.csv")
category_translation = pd.read_csv("../Data/product_category_name_translation.csv")

In [3]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[date_columns] = orders[date_columns].apply(pd.to_datetime)

reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])

order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

# Feature 1 : Purchase Year

In [4]:
orders["purchase_year"] = orders["order_purchase_timestamp"].dt.year

In [5]:
orders["purchase_year"].value_counts().sort_index()

purchase_year
2016      329
2017    45101
2018    54011
Name: count, dtype: int64

In [6]:
orders["purchase_month"] = orders["order_purchase_timestamp"].dt.month_name()

In [7]:
orders["purchase_month"].head()

0     October
1        July
2      August
3    November
4    February
Name: purchase_month, dtype: object

In [8]:
orders["purchase_day"] = orders["order_purchase_timestamp"].dt.day_name()

In [9]:
orders["purchase_day"].head()

0       Monday
1      Tuesday
2    Wednesday
3     Saturday
4      Tuesday
Name: purchase_day, dtype: object

In [10]:
orders["purchase_hour"] = orders["order_purchase_timestamp"].dt.hour

In [11]:
orders["purchase_hour"].describe()

count    99441.000000
mean        14.770829
std          5.326800
min          0.000000
25%         11.000000
50%         15.000000
75%         19.000000
max         23.000000
Name: purchase_hour, dtype: float64

In [12]:
orders["is_weekend"] = orders["purchase_day"].isin(["Saturday","Sunday"])

In [13]:
orders["is_weekend"].value_counts()

is_weekend
False    76594
True     22847
Name: count, dtype: int64

In [14]:
orders["delivery_time"] = (
    orders["order_delivered_customer_date"] -
    orders["order_purchase_timestamp"]
).dt.days

In [15]:
orders["delivery_time"].describe()

count    96476.000000
mean        12.094086
std          9.551746
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_time, dtype: float64

In [16]:
orders["estimated_delivery_time"] = (
    orders["order_estimated_delivery_date"] -
    orders["order_purchase_timestamp"]
).dt.days

In [17]:
orders[["delivery_time","estimated_delivery_time"]].head()

,delivery_time,estimated_delivery_time
0,8.0,15
1,13.0,19
2,9.0,26
3,13.0,26
4,2.0,12


In [18]:
orders["delivery_delay"] = (
    orders["order_delivered_customer_date"] -
    orders["order_estimated_delivery_date"]
).dt.days

In [19]:
orders["delivery_delay"].describe()

count    96476.000000
mean       -11.876881
std         10.183854
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delivery_delay, dtype: float64

In [20]:
orders["late_delivery"] = orders["delivery_delay"] > 0

In [21]:
orders["late_delivery"].value_counts()

late_delivery
False    92906
True      6535
Name: count, dtype: int64

# Creating Master Dataset

## Objective

The Olist dataset consists of multiple relational tables.

To perform meaningful business analysis, we will combine the required tables into a single analytical dataset using primary and foreign keys.

In [22]:
master_df = orders.merge(
    customers,
    on="customer_id",
    how="left"
)

In [23]:
master_df.shape

(99441, 21)

In [24]:
master_df = master_df.merge(
    order_items,
    on="order_id",
    how="left"
)

In [25]:
master_df.shape

(113425, 27)

In [26]:
master_df = master_df.merge(
    products,
    on="product_id",
    how="left"
)

In [27]:
master_df.shape

(113425, 35)

In [28]:
master_df = master_df.merge(
    sellers,
    on="seller_id",
    how="left"
)

In [29]:
master_df.shape

(113425, 38)

In [30]:
master_df = master_df.merge(
    payments,
    on="order_id",
    how="left"
)

In [31]:
master_df.shape

(118434, 42)

In [32]:
master_df = master_df.merge(
    reviews,
    on="order_id",
    how="left"
)

In [33]:
master_df.shape

(119143, 48)

In [34]:
master_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase_year,purchase_month,purchase_day,purchase_hour,is_weekend,delivery_time,estimated_delivery_time,delivery_delay,late_delivery,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,October,Monday,10,False,8.0,15,-8.0,False,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,9350.0,maua,SP,1.0,credit_card,1.0,18.12,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,October,Monday,10,False,8.0,15,-8.0,False,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,9350.0,maua,SP,3.0,voucher,1.0,2.00,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,October,Monday,10,False,8.0,15,-8.0,False,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,9350.0,maua,SP,2.0,voucher,1.0,18.59,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018,July,Tuesday,20,False,13.0,19,-6.0,False,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,31570.0,belo horizonte,SP,1.0,boleto,1.0,141.46,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018,August,Wednesday,8,False,9.0,26,-18.0,False,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,14840.0,guariba,SP,1.0,credit_card,3.0,179.12,e73b67b67587f7644d5bd1a52deb1b01,5.0,NaN,NaN,2018-08-18,2018-08-22 19:07:58


In [35]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119143 entries, 0 to 119142
Data columns (total 48 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       119143 non-null  object        
 1   customer_id                    119143 non-null  object        
 2   order_status                   119143 non-null  object        
 3   order_purchase_timestamp       119143 non-null  datetime64[ns]
 4   order_approved_at              118966 non-null  datetime64[ns]
 5   order_delivered_carrier_date   117057 non-null  datetime64[ns]
 6   order_delivered_customer_date  115722 non-null  datetime64[ns]
 7   order_estimated_delivery_date  119143 non-null  datetime64[ns]
 8   purchase_year                  119143 non-null  int32         
 9   purchase_month                 119143 non-null  object        
 10  purchase_day                   119143 non-null  object        
 11  

In [36]:
master_df.to_csv("../Data/master_dataset.csv", index=False)